# Siamese Network for Facial Recognition

This notebook builds a **Siamese neural network** for one-shot facial recognition using TensorFlow/Keras.

Instead of training a classifier to output a fixed set of identities, a Siamese network learns an
**embedding space** where images of the same person are close together and images of different
people are far apart. This makes it possible to recognize a new (unseen) image of a known person
using only a single reference photo, and to reject unknown people entirely.

**Pipeline overview:**
1. Generate a small synthetic face dataset (placeholder images per person).
2. Load, preprocess, and augment the images.
3. Build a CNN embedding network and wrap it in a Siamese architecture with a distance layer.
4. Generate positive (same-person) and negative (different-person) image pairs.
5. Train the network using contrastive loss.
6. Evaluate performance on a held-out set of pairs.
7. Build a "gallery" of known face embeddings and use nearest-neighbor matching to recognize new query images.


## 1. Imports

In [ ]:
import os
import random
import numpy as np
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


## 2. Generate a Synthetic Face Dataset

For demonstration/testing purposes, we generate placeholder "face" images rather than using real
photos: each simulated person is a colored circle ("face") with two darker circles ("eyes"), with
slight color/position/size jitter applied per image to simulate different photos of the same person.

Four synthetic people are created, each with 5 images, saved under `dataset/<person>/imgN.jpg`.


In [ ]:
np.random.seed(42)

people = ['person_1', 'person_2', 'person_3', 'person_4']
base_colors = [(200, 150, 150), (150, 200, 150), (150, 150, 200), (200, 200, 150)]

for person, base_color in zip(people, base_colors):
    folder = f'dataset/{person}'
    os.makedirs(folder, exist_ok=True)
    for i in range(5):
        img = Image.new('RGB', (105, 105), color=(240, 240, 240))
        draw = ImageDraw.Draw(img)

        # Jitter color/shape slightly per image, to simulate different photos of the same person
        jitter = np.random.randint(-15, 15, size=3)
        color = tuple(np.clip(np.array(base_color) + jitter, 0, 255))
        cx, cy = 52 + np.random.randint(-5, 5), 52 + np.random.randint(-5, 5)
        r = 35 + np.random.randint(-3, 3)
        draw.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)

        # Simple 'eyes'
        draw.ellipse([cx - 15, cy - 10, cx - 5, cy], fill=(50, 50, 50))
        draw.ellipse([cx + 5, cy - 10, cx + 15, cy], fill=(50, 50, 50))

        img.save(f'{folder}/img{i+1}.jpg')

print("Created dataset folders:")
for person in people:
    files = os.listdir(f'dataset/{person}')
    print(f"  {person}: {len(files)} images")


## 3. Load Dataset Structure

In [ ]:
def load_dataset_structure(dataset_path):
    """
    Scans a dataset directory of the form dataset/<person_name>/<image files>
    and returns a dict mapping each person's name to a list of their image file paths.
    """
    people_images = {}
    for person_name in sorted(os.listdir(dataset_path)):
        person_folder = os.path.join(dataset_path, person_name)
        if os.path.isdir(person_folder):
            image_files = [
                os.path.join(person_folder, f)
                for f in sorted(os.listdir(person_folder))
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ]
            people_images[person_name] = image_files
    return people_images


people_images = load_dataset_structure('dataset')
for person, images in people_images.items():
    print(f"{person}: {len(images)} images")


## 4. Train / Holdout Split

Before any augmentation, we set aside one image per person as a "holdout" — these images are never
seen during training and are used later to evaluate recognition on genuinely unseen photos.


In [ ]:
def split_holdout(people_images, holdout_per_person=1):
    """
    Splits each person's image paths into a training set and a small holdout set,
    shuffling first so the holdout image(s) are chosen randomly rather than always
    being e.g. the last file alphabetically.
    """
    train_paths, holdout_paths = {}, {}
    for person, paths in people_images.items():
        paths = paths.copy()
        random.shuffle(paths)
        holdout_paths[person] = paths[:holdout_per_person]
        train_paths[person] = paths[holdout_per_person:]
    return train_paths, holdout_paths


train_paths, holdout_paths = split_holdout(people_images, holdout_per_person=1)

print("Train paths per person:", {p: len(v) for p, v in train_paths.items()})
print("Holdout paths per person:", {p: len(v) for p, v in holdout_paths.items()})


## 5. Image Loading & Preprocessing

In [ ]:
def load_and_preprocess_image(image_path, target_size=(105, 105)):
    """
    Loads an image from disk, converts to RGB, resizes to the target size expected
    by the embedding network, and normalizes pixel values to the [0, 1] range.
    """
    img = Image.open(image_path).convert('RGB')
    img = img.resize(target_size)
    img_array = np.array(img) / 255.0
    return img_array


# Quick sanity check on a single image
test_img = load_and_preprocess_image(people_images['person_1'][0])
print(test_img.shape)                  # expect (105, 105, 3)
print(test_img.min(), test_img.max())  # expect ~0.0 to ~1.0


## 6. Data Augmentation

Since the dataset is very small (5 images per person), we apply simple augmentations to each image
to increase the effective size and diversity of the training set: a horizontal flip, a brightened
version, and a darkened version.


In [ ]:
def augment_image(img_array):
    """
    Generates 3 augmented variants of a single preprocessed image:
    a horizontal flip, a brighter version, and a darker version.
    Returns a list of augmented images (original is NOT included here).
    """
    augmented_versions = []

    # Horizontal flip
    flipped = np.fliplr(img_array)
    augmented_versions.append(flipped)

    # Brightness up
    brighter = np.clip(img_array * 1.2, 0, 1)
    augmented_versions.append(brighter)

    # Brightness down
    darker = np.clip(img_array * 0.8, 0, 1)
    augmented_versions.append(darker)

    return augmented_versions


test_img = load_and_preprocess_image(people_images['person_1'][0])
augmented = augment_image(test_img)

print("Number of augmented versions:", len(augmented))
print("Shape of each:", augmented[0].shape)


## 7. Build the Augmented Training Set

In [ ]:
def build_augmented_dataset(people_images, target_size=(105, 105)):
    """
    For every image path in `people_images`, loads and preprocesses the original image,
    then generates and appends its augmented variants. Returns a dict mapping each
    person to a list of (original + augmented) image arrays.
    """
    all_images = {}

    for person, paths in people_images.items():
        images_for_person = []
        for path in paths:
            original = load_and_preprocess_image(path, target_size)
            images_for_person.append(original)

            augmented = augment_image(original)
            images_for_person.extend(augmented)

        all_images[person] = images_for_person

    return all_images


all_images_train = build_augmented_dataset(train_paths)

for person, images in all_images_train.items():
    print(f"{person}: {len(images)} images (after augmentation)")


## 8. Preprocess the Holdout Set

The holdout images are preprocessed (resized/normalized) but **not** augmented — they represent
genuinely new photos the model should recognize at evaluation time.


In [ ]:
all_images_holdout = {
    p: [load_and_preprocess_image(path) for path in paths]
    for p, paths in holdout_paths.items()
}

for person, images in all_images_holdout.items():
    print(f"{person}: {len(images)} holdout images")


## 9. Embedding Network

This CNN maps a 105x105x3 face image to a 128-dimensional embedding vector. It is the shared
("twin") network used on both sides of the Siamese architecture — the same weights are applied to
both input images, so that similarity in the embedding space directly reflects visual similarity.


In [ ]:
def build_embedding_network(input_shape=(105, 105, 3)):
    """
    Builds the shared CNN "twin" network: 3 conv+pool blocks followed by a dense
    layer that produces a 128-dimensional embedding for an input face image.
    """
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(128, activation='relu')(x)

    embedding_model = Model(inputs, outputs, name='embedding_network')
    return embedding_model


embedding_net = build_embedding_network()
embedding_net.summary()


## 10. Siamese Architecture

Two images (A and B) are passed through the **same** embedding network (shared weights), and the
Euclidean distance between their resulting embeddings is computed. A small distance means the
network believes the two images are of the same person; a large distance means different people.


In [ ]:
input_a = layers.Input(shape=(105, 105, 3), name='image_a')
input_b = layers.Input(shape=(105, 105, 3), name='image_b')

# Both inputs go through the SAME embedding_net instance -> shared weights (this is
# what makes it a "Siamese" network)
embedding_a = embedding_net(input_a)
embedding_b = embedding_net(input_b)


def euclidean_distance(vectors):
    """Computes the Euclidean distance between two embedding vectors.
    A small epsilon (1e-9) is added under the square root for numerical stability."""
    emb_a, emb_b = vectors
    sum_squared = tf.reduce_sum(tf.square(emb_a - emb_b), axis=1, keepdims=True)
    return tf.sqrt(tf.maximum(sum_squared, 1e-9))


distance = layers.Lambda(euclidean_distance, output_shape=(1,))([embedding_a, embedding_b])

siamese_model = Model(inputs=[input_a, input_b], outputs=distance, name='siamese_network')
siamese_model.summary()


## 11. Generate Training & Holdout Pairs

The Siamese network is trained on **pairs** of images labeled either:
- `1` (positive pair — same person), or
- `0` (negative pair — different people)

`generate_pairs` builds balanced pairs from the augmented training set.
`generate_holdout_pairs` builds evaluation pairs using the never-seen holdout image for each
person against images from the training set, to test genuine generalization.


In [ ]:
def generate_pairs(all_images, num_pairs_per_person=20):
    """
    Builds a balanced set of training pairs for each person:
    - `num_pairs_per_person` positive pairs (two different images of the same person, label=1)
    - `num_pairs_per_person` negative pairs (one image of this person + one of a random
      different person, label=0)
    """
    pair_images_a, pair_images_b, labels = [], [], []
    people = list(all_images.keys())

    for person in people:
        images = all_images[person]

        # Positive pairs: two distinct images of the SAME person
        for _ in range(num_pairs_per_person):
            img1, img2 = random.sample(images, 2)
            pair_images_a.append(img1)
            pair_images_b.append(img2)
            labels.append(1)

        # Negative pairs: one image of this person + one image of a random different person
        other_people = [p for p in people if p != person]
        for _ in range(num_pairs_per_person):
            other_person = random.choice(other_people)
            img1 = random.choice(images)
            img2 = random.choice(all_images[other_person])
            pair_images_a.append(img1)
            pair_images_b.append(img2)
            labels.append(0)

    return np.array(pair_images_a), np.array(pair_images_b), np.array(labels)


def generate_holdout_pairs(all_images_holdout, all_images_train, num_negative_per_person=3):
    """
    Builds evaluation pairs using each person's held-out (never-trained-on) image:
    - 1 positive pair: holdout image vs. a training image of the SAME person
    - `num_negative_per_person` negative pairs: holdout image vs. training images of
      other people
    """
    pair_images_a, pair_images_b, labels = [], [], []
    people = list(all_images_holdout.keys())

    for person in people:
        holdout_img = all_images_holdout[person][0]

        # Positive pair: holdout image vs. a training image of the same person
        train_img_same = random.choice(all_images_train[person])
        pair_images_a.append(holdout_img)
        pair_images_b.append(train_img_same)
        labels.append(1)

        # Negative pairs: holdout image vs. training images of other people
        other_people = [p for p in people if p != person]
        for _ in range(num_negative_per_person):
            other_person = random.choice(other_people)
            train_img_diff = random.choice(all_images_train[other_person])
            pair_images_a.append(holdout_img)
            pair_images_b.append(train_img_diff)
            labels.append(0)

    return np.array(pair_images_a), np.array(pair_images_b), np.array(labels)


pairs_a, pairs_b, pair_labels = generate_pairs(all_images_train, num_pairs_per_person=20)
print("Total pairs:", len(pair_labels))


## 12. Contrastive Loss & Accuracy

The Siamese network is trained with **contrastive loss**, which:
- Pulls same-person embeddings together (penalizes large distance when label=1)
- Pushes different-person embeddings apart, up to a fixed `margin` (penalizes small
  distance when label=0, but only up to the margin — beyond that, no further penalty)

`contrastive_accuracy` provides a Keras metric that thresholds the predicted distance at
`margin / 2` to decide "same person" vs. "different person" during training.


In [ ]:
def contrastive_accuracy(margin=1.0):
    """Returns a Keras metric function that classifies a pair as 'same person' if the
    predicted distance is below margin/2, and compares against the true label."""
    def accuracy(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        threshold = margin / 2.0
        preds = tf.cast(tf.squeeze(y_pred) < threshold, tf.float32)
        return tf.reduce_mean(tf.cast(tf.equal(preds, y_true), tf.float32))
    return accuracy


def contrastive_loss(margin=1.0):
    """Returns the contrastive loss function:
    - For positive pairs (y_true=1): loss = distance^2 (pulls embeddings together)
    - For negative pairs (y_true=0): loss = max(margin - distance, 0)^2 (pushes
      embeddings apart, up to the margin)
    """
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        squared_pred = tf.square(y_pred)
        margin_square = tf.square(tf.maximum(margin - y_pred, 0))
        return tf.reduce_mean(y_true * squared_pred + (1 - y_true) * margin_square)
    return loss


## 13. Train the Siamese Network

In [ ]:
margin = 1.0
siamese_model.compile(
    optimizer='adam',
    loss=contrastive_loss(margin),
    metrics=[contrastive_accuracy(margin)]
)

history = siamese_model.fit(
    [pairs_a, pairs_b], pair_labels,
    validation_split=0.2, batch_size=16, epochs=10, verbose=2
)


## 14. Plot Training vs. Validation Loss

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Contrastive Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid(True)
plt.show()


## 15. Evaluate on Held-Out Pairs

We generate a fresh set of pairs using the holdout images (never seen during training) to properly
evaluate generalization, then compute standard classification metrics using a simple distance
threshold to decide "same person" vs. "different person".


In [ ]:
# Generate a fresh set of pairs, separate from training, to properly evaluate on unseen data
test_pairs_a, test_pairs_b, test_labels = generate_holdout_pairs(
    all_images_holdout, all_images_train, num_negative_per_person=3
)

# Shuffle these too, same reasoning as before (avoid any ordering bias)
test_indices = np.arange(len(test_labels))
np.random.shuffle(test_indices)
test_pairs_a = test_pairs_a[test_indices]
test_pairs_b = test_pairs_b[test_indices]
test_labels = test_labels[test_indices]

predicted_distances = siamese_model.predict([test_pairs_a, test_pairs_b])
predicted_distances = predicted_distances.flatten()

print("Distance range:", predicted_distances.min(), "to", predicted_distances.max())
print("Average distance for SAME-person pairs:", predicted_distances[test_labels == 1].mean())
print("Average distance for DIFFERENT-person pairs:", predicted_distances[test_labels == 0].mean())

threshold = 0.5   # we'll refine this in a moment

predicted_labels = (predicted_distances < threshold).astype(int)

accuracy = accuracy_score(test_labels, predicted_labels)
precision = precision_score(test_labels, predicted_labels)
recall = recall_score(test_labels, predicted_labels)
f1 = f1_score(test_labels, predicted_labels)

print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")


## 16. Build a Face Gallery & Recognition Function

For real-world recognition, we don't compare against pairs — instead we build a **gallery**: one
reference embedding per known person. A new query image is recognized by finding the closest
gallery embedding (nearest neighbor); if the closest distance still exceeds a threshold, the query
is classified as "Unknown" rather than forced into the closest match.


In [ ]:
def build_gallery(all_images, embedding_net):
    """Store one embedding per person, using their first image as reference."""
    gallery = {}
    for person, images in all_images.items():
        reference_image = images[0]                                 # pick one representative image
        reference_image = np.expand_dims(reference_image, axis=0)    # add batch dimension
        embedding = embedding_net.predict(reference_image, verbose=0)
        gallery[person] = embedding[0]                               # store as a plain 128-length vector
    return gallery


def recognize_face(query_image, gallery, embedding_net, threshold=0.595):
    """
    Embeds a query image and finds the closest match in the gallery by Euclidean
    distance. If the closest distance exceeds `threshold`, the query is classified
    as 'Unknown' rather than being forced into the nearest (but still dissimilar) match.
    """
    query_image = np.expand_dims(query_image, axis=0)
    query_embedding = embedding_net.predict(query_image, verbose=0)[0]

    best_match = None
    best_distance = float('inf')

    for person, stored_embedding in gallery.items():
        distance = np.linalg.norm(query_embedding - stored_embedding)  # Euclidean distance
        if distance < best_distance:
            best_distance = distance
            best_match = person

    if best_distance < threshold:
        return best_match, best_distance
    else:
        return "Unknown", best_distance


## 17. Visualize a Recognition Result

In [ ]:
def display_recognition_result(query_image_path, gallery, all_images, embedding_net,
                                target_size=(105, 105), threshold=0.595):
    """
    Runs face recognition on a query image and displays it side-by-side with the
    gallery's reference image for the predicted person (or a blank placeholder if
    the query is classified as Unknown).
    """
    query_image = load_and_preprocess_image(query_image_path, target_size)
    predicted_person, distance = recognize_face(query_image, gallery, embedding_net, threshold)

    # Get a reference image of the predicted person to show side-by-side (if matched)
    if predicted_person != "Unknown":
        reference_image = all_images[predicted_person][0]
    else:
        reference_image = np.ones_like(query_image)  # blank placeholder if unknown

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow(query_image)
    axes[0].set_title("Query Image")
    axes[0].axis('off')

    axes[1].imshow(reference_image)
    axes[1].set_title(f"Best Match:\n{predicted_person}")
    axes[1].axis('off')

    plt.suptitle(f"Distance: {distance:.4f}  |  Decision: {predicted_person}")
    plt.tight_layout()
    plt.show()


## 18. Run an End-to-End Recognition Example

In [ ]:
gallery = build_gallery(all_images_train, embedding_net)

# Test with a real holdout image path (never seen during training)
query_image = holdout_paths['person_2'][0]

display_recognition_result(query_image, gallery, all_images_train, embedding_net, threshold=0.595)
